<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/course/07%20-%20Neural%20Networks%20and%20CNNs/07a_neural_networks.ipynb)

# 07a · 神经网络基础：从单个神经元到多层感知机与不确定性估计

在前四讲中，我们围绕手工设计的化学描述符（RDKit 物理化学性质、Morgan 指纹）构建了线性回归、kNN、随机森林以及高斯过程回归模型。在约九千个分子的 AqSolDB 水溶解度数据集上，随机森林将测试集 RMSE 推进到了约 1.17 $\log S$ 单位。

从本讲开始，课程进入深度学习部分。读者可能会问：既然随机森林在表格特征上已经表现稳健，为什么材料与化学信息学还需要人工神经网络（Artificial Neural Networks）？原因主要体现在三个层面：

1. **表征学习（Representation Learning）**：传统机器学习依赖人工预先定义特征，而深度神经网络能够直接从原始数据结构——例如显微图像像素阵列（`07b`）、SMILES 序列（`08`）或分子图与三维原子坐标（`10`）——中端到端地自动提取任务相关的特征表示。
2. **模块化架构与可扩展性**：神经网络像搭积木一样，通过组合线性层、卷积层、注意力层或消息传递层即可适配完全不同的物理约束（如平移不变性、置换不变性），并借助小批量梯度下降轻松扩展到百万级样本。
3. **生成与逆向设计能力**：基于神经网络的变分自编码器（VAE）和扩散模型能够学习连续的潜在化学空间，实现从“给定结构预测性质”到“给定目标性质逆向生成分子或微结构”的跨越。

不过，深度学习并非在所有场景下都优于经典方法：**在中小规模、固定维度的手工表格特征上，多层感知机（MLP）往往与随机森林难分伯仲，甚至更容易过拟合。** 本讲我们将使用纯 PyTorch 从零写出完整的神经网络训练流程，并在与第 02 讲完全一致的 AqSolDB 划分上亲自验证这一结论。

### 本节大纲

1. 线性模型的局限、人工神经元与非线性激活函数
2. PyTorch 核心基石：张量（Tensor）与自动微分（Autograd）
3. 手写梯度下降：从头拟合 Yalkowsky 溶解度线性项
4. Python 面向对象基础（`class`）与 `nn.Module` 封装规范
5. 搭建与训练多层感知机（训练/验证/测试划分、标准化、`DataLoader` 与 Adam 循环）
6. 过拟合诊断、三类正则化对策与 MC Dropout 不确定性估计
7. 学习率动力学与二分类交叉熵扩展

In [ ]:
# Colab 环境初始化（本地运行时自动跳过）
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "course", "07 - Neural Networks and CNNs"))
    get_ipython().run_line_magic("pip", "install -q rdkit")

In [ ]:
import copy
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
RDLogger.DisableLog("rdApp.*")
print("PyTorch version:", torch.__version__)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


神经网络训练包含多处随机源：各层权重的随机初始化、小批量采样时的顺序打乱，以及部分底层算子的并行归约顺序。为了保证实验结果完全可复现，我们在开始前统一固定随机种子并启用确定性算法。

In [ ]:
def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.use_deterministic_algorithms(True)

set_seed(0)

In [ ]:
DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"

df = pd.read_csv(DATA_FILE)
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)

feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT",
]
X = df[feature_cols].values
y = df["Solubility"].values
print(f"有效分子数: {len(df)} | 特征维度: {X.shape[1]}")

---
## 1. 人工神经元与非线性激活函数

### 1.1 从线性回归到多层感知机

单个**人工神经元（Neuron）**的数学运算分为两步：

1. **仿射变换（加权求和加偏置）**：
   $$z = \sum_{i=1}^{d} w_i x_i + b = \mathbf{w}^\top \mathbf{x} + b$$
   这与第 02 讲中的多元线性回归完全一致。
2. **非线性激活（Activation）**：将标量 $z$ 送入非线性函数 $\sigma(\cdot)$，得到神经元输出 $a = \sigma(z)$。

当我们将多个神经元并排排列成一层（隐藏层，Hidden Layer），并把多层首尾相接堆叠起来时，就构成了**多层感知机（Multilayer Perceptron, MLP）**：

```
输入层               隐藏层 1             隐藏层 2             输出层
(17 个分子描述符)    (64 个神经元)        (64 个神经元)        (预测 logS)
  x1  ──┐
  x2  ──┼──►        ○ ○ ○ … ○    ──►    ○ ○ ○ … ○    ──►      ○
  …     │           h1 = σ(W1 x + b1)   h2 = σ(W2 h1 + b2)    y = W3 h2 + b3
  x17 ──┘
```

### 1.2 为什么必须引入非线性激活函数？

如果不加非线性激活函数 $\sigma$，直接将两层甚至十层线性变换串联起来会发生什么？根据矩阵乘法结合律：

$$\mathbf{W}_2 (\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = (\mathbf{W}_2 \mathbf{W}_1) \mathbf{x} + (\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2) = \tilde{\mathbf{W}} \mathbf{x} + \tilde{\mathbf{b}}$$

无论堆叠多少层，复合后的映射仍然严格等价于**单层线性回归**，无法拟合任何弯曲的物理化学响应面。正是每一层之后的非线性激活函数打破了线性叠加，赋予了神经网络逼近任意连续函数的能力（万能逼近定理）。

下面绘制三种最常见的激活函数：**ReLU**（Rectified Linear Unit，$\max(0, z)$）、**Sigmoid**（$1 / (1 + e^{-z})$）与 **Tanh**（$\tanh(z)$）。

In [ ]:
z = torch.linspace(-4, 4, 200)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(z.tolist(), torch.relu(z).tolist(), lw=2, label="ReLU: max(0, z)")
ax.plot(z.tolist(), torch.sigmoid(z).tolist(), lw=2, label="Sigmoid: 1 / (1 + exp(-z))")
ax.plot(z.tolist(), torch.tanh(z).tolist(), lw=2, label="Tanh: tanh(z)")
ax.axhline(0, color="gray", lw=0.6)
ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("Pre-activation z")
ax.set_ylabel("Activation sigma(z)")
ax.set_title("Common Nonlinear Activation Functions")
ax.legend()
plt.tight_layout()
plt.show()

在现代深度网络中，**ReLU** 是隐藏层的默认首选：它在正半轴导数恒为 1，计算开销极低，且有效缓解了 Sigmoid/Tanh 在两端饱和区导数趋近于 0 所引起的**梯度消失（Vanishing Gradient）**问题。

---
## 2. PyTorch 核心基石：张量与自动微分

### 2.1 张量（Tensor）

PyTorch 的基础数据结构是 `torch.Tensor`。你可以将它理解为支持 GPU 硬件加速与自动求导的 NumPy 多维数组，其索引、广播与矩阵乘法语法与 NumPy 高度一致。由于深度神经网络对显存带宽敏感，模型权重与特征张量通常采用 32 位浮点数（`torch.float32`）。

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print("Tensor a:\n", a)
print("Shape:", a.shape, "| Dtype:", a.dtype)
print("Column sum (dim=0):", a.sum(dim=0))
print("Matrix product a @ a.T:\n", a @ a.T)

# NumPy 数组与 PyTorch 张量的转换
x_np = np.array([1.5, 2.5])
x_t = torch.tensor(x_np, dtype=torch.float32)
print("NumPy -> Tensor -> NumPy:", x_t, "->", x_t.numpy())

### 2.2 自动微分（Autograd）

训练神经网络的本质是通过微积分的链式法则计算损失函数 $L$ 对每一个模型参数的偏导数 $\partial L / \partial w$。当参数量达到数万乃至数亿时，手工推导解析导数是不现实的。

在 PyTorch 中，只需为参数张量指定 `requires_grad=True`，框架就会在前向计算过程中自动构建动态计算图（Computational Graph）。调用标量输出的 `.backward()` 方法后，反向传播算法会自动沿着计算图回溯，并将偏导数存入各叶节点张量的 `.grad` 属性中。

例如，对于二次函数 $f(x) = x^2 + 3x$，解析导数为 $f'(x) = 2x + 3$，在 $x = 2$ 处应有 $f(2) = 10$、$f'(2) = 7$：

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x ** 2 + 3 * x
f.backward()

print("f(2) =", f.item())
print("df/dx at x=2 (autograd):", x.grad.item())

---
## 3. 手写梯度下降：拟合 Yalkowsky 溶解度线性项

有了自动求导，我们就能用五步循环实现**梯度下降（Gradient Descent）**：

1. **初始化参数**：给定初始猜测值（例如 $w = 0, b = 0$）；
2. **前向传播与损失计算**：根据当前参数计算预测值 $\hat{y}$，并评估均方误差损失 $L = \frac{1}{N}\sum (\hat{y}_i - y_i)^2$；
3. **反向传播求梯度**：调用 `loss.backward()` 计算 $\partial L / \partial w$ 与 $\partial L / \partial b$；
4. **沿负梯度方向更新参数**：
   $$w \leftarrow w - \eta \frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$
   其中步长因子 $\eta > 0$ 称为**学习率（Learning Rate）**；
5. **梯度清零**：PyTorch 默认会累加 `.grad` 中的梯度，因此在进入下一轮迭代前必须调用 `.grad.zero_()`。

我们用这五步循环来拟合最简单的单变量模型：$\log S = w \cdot \log P + b$，并将迭代收敛结果与最小二乘解析解 `np.polyfit` 对比。

In [ ]:
logp = torch.tensor(df["MolLogP"].values, dtype=torch.float32)
logs = torch.tensor(y, dtype=torch.float32)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.01

losses = []
for step in range(300):
    pred = w * logp + b
    loss = ((pred - logs) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()
    losses.append(loss.item())

print(f"梯度下降 (300 步): w = {w.item():.3f}, b = {b.item():.3f}")
w_ols, b_ols = np.polyfit(df["MolLogP"], y, 1)
print(f"最小二乘解析解:    w = {w_ols:.3f}, b = {b_ols:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(losses, lw=2)
ax.set_xlabel("Iteration Step")
ax.set_ylabel("MSE Loss")
ax.set_yscale("log")
ax.set_title("Gradient Descent Convergence on logS vs. MolLogP")
plt.tight_layout()
plt.show()

可以看到，梯度下降从 $(w, b) = (0, 0)$ 出发，仅靠一步步顺着负梯度微调，最终收敛到了与最小二乘解析解几乎一致的参数（$w \approx -0.41, b \approx -2.06$）。

顺便值得注意的是，第 02c 讲介绍的理想 Yalkowsky 方程 $\log S \approx 0.5 - \log P - 0.01(\text{MP} - 25)$ 中 $\log P$ 的理论斜率为 $-1$，而 AqSolDB 实测拟合斜率平缓到了 $-0.41$。这主要源于两方面物理原因：一是数据集中包含大量不遵循中性有机分子分配规律的盐类与离子型化合物；二是输入特征 `MolLogP` 本身是 RDKit 基团贡献法的估算值，自变量含有的随机误差会产生统计上的“衰减偏误（Attenuation Bias）”，把回归斜率向 0 拉平。

如果将上面代码中的 `learning_rate` 改为 `0.1`，你会发现损失不仅不下降，反而迅速越过谷底并在两侧剧烈震荡直至溢出为 `nan`（梯度爆炸/发散）；反之若设为 `1e-4`，则 300 步后仍远未收敛。整个深度学习的训练循环与这段 15 行代码本质相同，只是待优化参数从 2 个扩展到了数千乃至数百万个。

---
## 4. Python 的类（`class`）与 `nn.Module` 封装规范

在 PyTorch 中，所有神经网络模型都通过面向对象编程中的**类（`class`）**来组织。类定义了一类对象的**属性（数据状态）**与**方法（行为函数）**：

- `__init__(self, ...)` 是构造函数，在创建实例时自动执行，用于绑定对象的初始属性；
- 类中的普通函数以 `self` 作为第一个参数，代表当前实例自身。

下面我们先用一个表示化学分子的 `Molecule` 类来直观展示类的定义与实例化：

In [ ]:
class Molecule:
    def __init__(self, name, smiles):
        self.name = name
        self.smiles = smiles
        self.mol = Chem.MolFromSmiles(smiles)

    def n_heavy_atoms(self):
        return self.mol.GetNumHeavyAtoms()

    def describe(self):
        return f"{self.name} ({self.smiles}): {self.n_heavy_atoms()} heavy atoms"

thiophene = Molecule("thiophene", "c1ccsc1")
pentacene = Molecule("pentacene", "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1")
print(thiophene.describe())
print(pentacene.describe())

面向对象中的另一个关键机制是**继承（Inheritance）**：`class Child(Parent)` 表示子类自动继承父类的全部属性与方法，只需编写自己新增或重载的逻辑。

PyTorch 的所有网络模型均继承自基类 `nn.Module`。`nn.Module` 在底层自动完成了可训练参数的递归收集、梯度追踪、训练/评估模式切换以及权重序列化保存，我们只需在子类中实现两个方法：
- `__init__`：首先调用 `super().__init__()`，随后声明网络包含哪些子层（如 `nn.Linear`、`nn.ReLU`）；
- `forward(self, x)`：定义输入张量 `x` 如何依次流经各层得到预测输出。

### 练习 07a-1：扩展自定义类的方法

请为上面的 `Molecule` 类新增一个方法 `n_aromatic_rings(self)`，返回分子中的芳香环数目（提示：导入 `from rdkit.Chem import Descriptors`，调用 `Descriptors.NumAromaticRings(self.mol)`），并对噻吩（`thiophene`）与并五苯（`pentacene`）分别打印输出。

In [ ]:
# 练习 07a-1：在此编写代码


---
## 5. 搭建并训练第一个多层感知机（MLP）

### 5.1 训练集、验证集与测试集的严格划分

为了与第 02 讲的线性回归、kNN 和随机森林直接横向对比，我们首先使用完全相同的随机种子（`test_size=0.2, random_state=42`）切分出 20% 的独立**测试集**。

在神经网络训练过程中，我们需要持续监控模型何时开始过拟合（决定早停轮次），并比较不同网络宽度、Dropout 比例与学习率。正如第 02b 讲所强调的，任何超参数选择或早停决策都绝不能偷看测试集。因此，我们从剩余 80% 的开发集中再划分出 10% 作为**验证集（Validation Set）**：

| 数据子集 | 核心职责 | 样本占比 |
|---|---|---|
| **训练集（Train）** | 计算梯度并更新网络权重 | 72% (6825) |
| **验证集（Validation）** | 监控过拟合、选择最佳训练轮次（Early Stopping）与超参数 | 8% (759) |
| **测试集（Test）** | 所有训练与调参结束后**仅评估一次**，报告最终泛化指标 | 20% (1896) |

同时，不同描述符的量纲跨度极大（例如分子复杂度描述符 `BertzCT` 常达上千，而氢键供体数 `NumHDonors` 仅为 $0\sim 5$）。若直接送入神经网络，大数值特征对应的梯度分量将主导整个参数更新方向，导致优化轨迹剧烈震荡甚至无法收敛。因此，我们必须先在**训练集**上拟合 `StandardScaler`，再对三个集合做标准化变换并转为 `float32` 张量。

In [ ]:
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_trval, y_trval, test_size=0.1, random_state=42)
print(f"样本划分 -> 训练集: {len(X_train)} | 验证集: {len(X_valid)} | 测试集: {len(X_test)}")

# 仅在训练集上拟合标准化器，防止验证/测试集统计量泄漏
scaler = StandardScaler().fit(X_train)

def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)

Xtr, Xva, Xte = (to_tensor(scaler.transform(a)) for a in (X_train, X_valid, X_test))
ytr, yva, yte = (to_tensor(a) for a in (y_train, y_valid, y_test))

### 5.2 小批量迭代（Mini-batch）与 `DataLoader`

第 3 节手写梯度下降时，每更新一次参数都遍历了全部样本（批量梯度下降，Batch Gradient Descent）。在大数据集与非凸损失曲面上，更标准的做法是**小批量随机梯度下降（Mini-batch SGD）**：每次从打乱后的训练集中抽取一小批样本（例如 `batch_size=64`）计算梯度并更新一次参数。不仅单步计算极快，而且小批量引入的梯度估计噪声有助于模型跳出尖锐的局部极小值。

把整个训练集的所有小批量完整遍历一遍，称为一个 **Epoch（训练轮次）**。PyTorch 的 `TensorDataset` 与 `DataLoader` 帮我们自动完成了分批与每轮随机洗牌（`shuffle=True`）。

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

xb, yb = next(iter(train_loader))
print("单个 Mini-batch 的 X 形状:", xb.shape, "| y 形状:", yb.shape)
print(f"每个 Epoch 包含 {len(train_loader)} 个 Mini-batch（即每轮更新 {len(train_loader)} 次参数）")

### 5.3 定义多层感知机架构 `SolubilityMLP`

我们定义一个包含两个隐藏层的 MLP：输入层（17 维描述符）$\to$ 64 维线性层 + ReLU $\to$ 64 维线性层 + ReLU $\to$ 1 维输出层（预测 $\log S$）。同时在每个隐藏层后预留 `nn.Dropout(dropout)` 接口（默认 `dropout=0.0` 时不改变任何激活值，供第 6 节正则化实验使用）。

在运行下方代码前，不妨先心算一下该网络的参数总数：线性层 `nn.Linear(d_in, d_out)` 包含 $d_{\text{in}} \times d_{\text{out}}$ 个权重和 $d_{\text{out}}$ 个偏置，因此三层共有 $(17\times 64 + 64) + (64\times 64 + 64) + (64\times 1 + 1) = 5377$ 个可训练参数。

In [ ]:
class SolubilityMLP(nn.Module):
    def __init__(self, n_features, hidden_size=64, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)  # 将形状 (batch, 1) 压缩为 (batch,)

set_seed(0)
model = SolubilityMLP(n_features=17)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("可训练参数总数:", n_params)

对比第 02 讲仅含 18 个参数的线性回归，这个双层 MLP 拥有 5377 个参数（而训练集分子数为 6825）。参数量的激增赋予了网络强大的非线性拟合能力，但也意味着如果不加约束，模型很容易“死记硬背”训练集中的噪声。

### 5.4 标准训练循环与基线评估

工程实践中，我们不再手写 `w -= lr * w.grad`，而是使用 PyTorch 内置的损失函数 `nn.MSELoss()` 与自适应动量优化器 `torch.optim.Adam`（Adam 会根据每个参数历史梯度的一阶矩和二阶矩自动调整有效步长，收敛比朴素梯度下降平稳得多）。

| 第 3 节手写循环 | 标准 `nn.Module` 训练循环 |
|---|---|
| `pred = w * logp + b` | `pred = model(xb)` |
| `loss = ((pred - y)**2).mean()` | `loss = loss_fn(pred, yb)` |
| `w.grad.zero_()` | `optimizer.zero_grad()` |
| `loss.backward()` | `loss.backward()` |
| `w -= lr * w.grad` | `optimizer.step()` |

下面编写通用的训练与验证辅助函数，并在训练集上训练 `SolubilityMLP` 共 100 个 Epoch。

In [ ]:
loss_fn = nn.MSELoss()

def train_one_epoch(model, loader, optimizer):
    model.train()  # 开启训练模式（Dropout 生效）
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def rmse(model, X, y):
    model.eval()   # 开启评估模式（Dropout 关闭）
    with torch.no_grad():
        return torch.sqrt(loss_fn(model(X), y)).item()

def fit(model, n_epochs=100, lr=1e-3, weight_decay=0.0):
    """训练模型 n_epochs 轮，记录每轮训练/验证 RMSE，并保存验证集表现最佳的模型参数快照。"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "valid": []}
    best_valid, best_state = float("inf"), None
    for epoch in range(n_epochs):
        train_one_epoch(model, train_loader, optimizer)
        history["train"].append(rmse(model, Xtr, ytr))
        history["valid"].append(rmse(model, Xva, yva))
        if history["valid"][-1] < best_valid:
            best_valid = history["valid"][-1]
            best_state = copy.deepcopy(model.state_dict())
    return history, best_state

def plot_history(history, title=""):
    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    ax.plot(history["train"], lw=1.8, label="Train RMSE")
    ax.plot(history["valid"], lw=1.8, label="Validation RMSE")
    best = int(np.argmin(history["valid"]))
    ax.axvline(best, color="gray", ls=":", label=f"Best Epoch ({best})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("RMSE (logS)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
set_seed(0)
mlp = SolubilityMLP(n_features=17)
history, best_state = fit(mlp, n_epochs=100)
plot_history(history, "Baseline MLP (64x64) on 17 Descriptors")
print(f"第 100 轮 -> 训练 RMSE: {history['train'][-1]:.2f} | 验证 RMSE: {history['valid'][-1]:.2f}")
print(f"测试集 RMSE: {rmse(mlp, Xte, yte):.2f} (对照第 02 讲: 线性回归 1.64, 缩放 kNN 1.24, 随机森林 1.17)")

In [ ]:
mlp.eval()
with torch.no_grad():
    y_test_list = yte.tolist()
    y_pred_list = mlp(Xte).tolist()

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(y_test_list, y_pred_list, s=6, alpha=0.4)
ax.plot([-13, 3], [-13, 3], "k--", lw=1)
ax.set_xlabel("Measured logS")
ax.set_ylabel("Predicted logS")
ax.set_title("MLP (64x64) Parity Plot on Test Set")
plt.tight_layout()
plt.show()

在相同的 17 个手工描述符上，64×64 MLP 的测试集 RMSE 约为 1.19，与第 02 讲的随机森林（1.17）几乎持平。这印证了我们在开篇提出的观点：**当输入特征已经固定为低维手工描述符时，单纯将模型替换为多层感知机并不会带来性能飞跃。**

---
## 6. 过拟合诊断、三类正则化对策与 MC Dropout 不确定性

### 6.1 盲目扩大网络容量带来的严重过拟合

如果我们将每个隐藏层的神经元数目从 64 扩大 8 倍至 512（参数量从 5,377 暴增约 50 倍至 272,385），并训练 150 轮，结果会变好吗？

In [ ]:
set_seed(0)
big_mlp = SolubilityMLP(n_features=17, hidden_size=512)
print("大容量 MLP 参数总数:", sum(p.numel() for p in big_mlp.parameters()))
big_history, big_best_state = fit(big_mlp, n_epochs=150)
plot_history(big_history, "Overfitting in Large MLP (512x512, No Regularization)")
print(f"第 150 轮 -> 训练 RMSE: {big_history['train'][-1]:.2f} | 验证 RMSE: {big_history['valid'][-1]:.2f}")

学习曲线呈现出教科书式的**过拟合（Overfitting）**：训练 RMSE 一路下探到 0.57，而验证 RMSE 在前 20 轮触底后便持续抬头恶化到 1.41 以上。拥有 27 万个参数的网络对仅含 6825 个样本的训练集产生了严重记忆效应。

抑制神经网络过拟合有三种最经典且互补的技术：

### 6.2 对策一：早停（Early Stopping）

最直接的对策是在验证集误差达到最低点的那一轮及时停止，并回滚加载当时的模型权重（`best_state`）。

In [ ]:
print(f"第 150 轮末尾模型 测试 RMSE: {rmse(big_mlp, Xte, yte):.2f}")
big_mlp.load_state_dict(big_best_state)
print(f"早停最佳轮次模型   测试 RMSE: {rmse(big_mlp, Xte, yte):.2f}")

### 6.3 对策二与三：Dropout 与权重衰减（Weight Decay）

- **Dropout（随机失活）**：在训练阶段的每次前向传播中，以概率 $p$（如 $p=0.2$）独立随机地将隐藏层神经元输出置零，并将保留的神经元按 $1/(1-p)$ 缩放。这迫使网络不能过度依赖少数特定神经元的协同记忆，相当于在训练指数级多个共享权重的子网络集成。
- **权重衰减（Weight Decay / $L_2$ 正则化）**：在损失函数中加入权重平方和惩罚项 $\lambda \|\mathbf{w}\|_2^2$，与第 02a 讲的岭回归（Ridge）同源，限制权重幅值过度膨胀。

我们在 512×512 的大网络上同时开启 `dropout=0.2` 与 `weight_decay=1e-4` 重新训练 150 轮：

In [ ]:
set_seed(0)
reg_mlp = SolubilityMLP(n_features=17, hidden_size=512, dropout=0.2)
reg_history, reg_best_state = fit(reg_mlp, n_epochs=150, weight_decay=1e-4)
plot_history(reg_history, "Large MLP (512x512) + Dropout (0.2) + Weight Decay (1e-4)")
print(f"第 150 轮 -> 训练 RMSE: {reg_history['train'][-1]:.2f} | 验证 RMSE: {reg_history['valid'][-1]:.2f}")
reg_mlp.load_state_dict(reg_best_state)
print(f"正则化 + 早停后测试 RMSE: {rmse(reg_mlp, Xte, yte):.2f}")

加上 Dropout 与权重衰减后，验证误差曲线在 150 轮内始终保持平稳（最终验证 RMSE 从 1.41 降至约 1.15）。但汇总所有模型在同一测试集上的表现可以发现：

| 模型配置 | 测试集 RMSE ($\log S$) |
|---|---|
| 随机森林（第 02a 讲，17 个描述符） | 1.17 |
| MLP 64×64 | 1.19 |
| MLP 512×512（早停） | 1.17 |
| MLP 512×512（Dropout + 权重衰减 + 早停） | 1.17 |

即便将参数量扩大 50 倍并配齐正则化手段，测试集 RMSE 依然停留在 1.17。原因很明确：**当前的性能天花板受限于 17 个手工描述符的信息量（例如缺乏晶格能与熔点相关的三维堆积信息），而不在于回归器的非线性拟合能力。**

### 6.4 延伸应用：Monte Carlo Dropout（MC Dropout）不确定性估计

在第 04 讲贝叶斯优化中，我们学习了高斯过程回归（GPR）如何天然输出预测均值 $\mu(\mathbf{x})$ 与不确定性标准差 $\sigma(\mathbf{x})$。那么在使用深度神经网络作为代理模型时，如何低成本地获取预测不确定性？

Gal 与 Ghahramani（ICML 2016）证明：在**推理阶段故意保持 Dropout 开启**（即调用 `model.train()`），对同一个输入样本重复进行 $T$ 次随机前向传播，就等价于从近似贝叶斯后验分布中采样 $T$ 个子网络：

$$\hat{\mu}(\mathbf{x}) = \frac{1}{T}\sum_{t=1}^{T} f_{\hat{\mathbf{w}}_t}(\mathbf{x}), \qquad \hat{\sigma}(\mathbf{x}) = \sqrt{\frac{1}{T}\sum_{t=1}^{T}\left(f_{\hat{\mathbf{w}}_t}(\mathbf{x}) - \hat{\mu}(\mathbf{x})\right)^2}$$

这种方法被称为 **MC Dropout**。下面我们利用刚刚训练好的 `reg_mlp` 对测试集分子做 $T=50$ 次 MC Dropout 采样，检验高不确定性样本是否确实对应更大的实际预测误差。

In [ ]:
def mc_dropout_predict(model, X, n_samples=50):
    model.train()  # 关键：在推理时保持 Dropout 激活以进行蒙特卡洛采样
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X).unsqueeze(0))
    preds = torch.cat(preds, dim=0)  # (n_samples, N)
    return preds.mean(dim=0).numpy(), preds.std(dim=0).numpy()

set_seed(0)
mc_mean, mc_std = mc_dropout_predict(reg_mlp, Xte, n_samples=50)
abs_err = np.abs(mc_mean - yte.numpy())

# 按 MC Dropout 不确定性标准差分为低、中、高三组，比较各组的实际 RMSE
quantiles = np.quantile(mc_std, [0.0, 1/3, 2/3, 1.0])
print("MC Dropout 不确定性分组验证:")
for i, label in enumerate(["低不确定性 (Bottom 1/3)", "中不确定性 (Middle 1/3)", "高不确定性 (Top 1/3)   "]):
    mask = (mc_std >= quantiles[i]) & (mc_std <= quantiles[i + 1])
    group_rmse = np.sqrt(np.mean((mc_mean[mask] - yte.numpy()[mask]) ** 2))
    print(f"  {label}: 平均预测 std = {mc_std[mask].mean():.3f} | 实际测试 RMSE = {group_rmse:.3f}")

可以看到，随着 MC Dropout 给出的预测标准差从低到高递增，分子的实际预测 RMSE 也显著上升。这为我们提供了一种无需修改网络架构即可评估深度学习模型认知不确定性（Epistemic Uncertainty）的实用工具。

### 练习 07a-2：对比不同非线性激活函数

请编写一个支持自定义激活函数的新类 `FlexibleMLP(nn.Module)`（或在 `SolubilityMLP` 中增加 `activation=nn.ReLU` 参数），在 64×64 的网络结构下分别使用 `nn.ReLU` 和 `nn.Tanh` 训练 100 轮，打印两者在最后一轮和早停后的测试集 RMSE。

In [ ]:
# 练习 07a-2：在此编写代码


---
## 7. 学习率动力学与二分类扩展

### 7.1 学习率对收敛轨迹的影响

学习率（`lr`）是训练神经网络最重要的超参数。下面我们在 64×64 MLP 上对比三个跨数量级的学习率：`1e-1`、`1e-3`（Adam 默认值）和 `1e-5`，各训练 30 轮。

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.6))
for lr in [1e-1, 1e-3, 1e-5]:
    set_seed(0)
    h, _ = fit(SolubilityMLP(n_features=17), n_epochs=30, lr=lr)
    ax.plot(h["valid"], lw=1.8, label=f"lr = {lr:g}")
    print(f"lr = {lr:g}: 第 30 轮验证集 RMSE = {h['valid'][-1]:.2f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation RMSE (logS)")
ax.set_yscale("log")
ax.set_title("Effect of Learning Rate on Adam Optimization")
ax.legend()
plt.tight_layout()
plt.show()

三条曲线清晰展示了学习率选择的经验准则：
- `lr = 1e-5` 步长过小，30 轮后验证 RMSE 仍停留在 3.0 以上，甚至劣于直接预测训练集均值（RMSE 2.42）；
- `lr = 1e-3` 是 Adam 优化器的黄金起点，下降平滑且迅速收敛至 1.18 左右；
- `lr = 1e-1` 步长过大，参数在极小值附近来回跨越，导致验证曲线明显抖动且收敛精度受损。

### 练习 07a-3：将 MLP 改造为水溶解度二分类器

神经网络只需更换损失函数即可无缝切换为分类模型。请将任务改为第 02c 讲的“判断分子是否难溶（$\log S < -4$）”：
1. 将训练、验证、测试集的标签转换为浮点型 0/1 张量：`y_cls = (y < -4).astype(float)`；
2. 保持 `SolubilityMLP(n_features=17)` 架构不变（此时最后一层输出的实数称为 **Logit**）；
3. 使用内部集成 Sigmoid 数值稳定计算的二分类交叉熵损失 `nn.BCEWithLogitsLoss()` 训练 50 轮；
4. 预测正类概率为 `torch.sigmoid(model(X))`，调用 `sklearn.metrics.roc_auc_score` 计算测试集 ROC-AUC，并与第 02c 讲的随机森林分类器（ROC-AUC $\approx 0.953$）对比。

In [ ]:
# 练习 07a-3：在此编写代码


---
## 本节小结

| 核心概念 | PyTorch 实现与物理/算法要点 |
|---|---|
| **人工神经元与激活函数** | 线性仿射变换 + 非线性激活（`nn.ReLU`）；若无非线性激活，多层堆叠退化为单层线性回归 |
| **张量与自动微分** | `torch.tensor(..., dtype=torch.float32)`；`requires_grad=True` 追踪计算图，`.backward()` 自动求导 |
| **模型封装与小批量训练** | 继承 `nn.Module` 实现 `__init__` 与 `forward`；配合 `DataLoader` 执行 `zero_grad -> backward -> step` |
| **过拟合与正则化三剑客** | 验证集早停（Early Stopping）、随机失活（`nn.Dropout`）、权重衰减（`weight_decay`） |
| **MC Dropout 不确定性** | 推理阶段保持 `model.train()` 多次采样计算均值与标准差，高标准差区域对应更高的外推风险 |
| **表格数据的模型瓶颈** | 在相同的 17 个手工描述符上，MLP 与随机森林精度相当（RMSE $\approx 1.17$）；深度学习的真正威力在于直接处理图像、序列与分子图 |

在下一节 `07b_cnn_microstructure.ipynb` 中，我们将离开手工表格特征，进入深度学习真正大放异彩的第一个领域——使用**卷积神经网络（CNN）**直接从二维材料显微图像（SEM/金相微结构）中识别相貌并定量提取相体积分数。